In [1]:
%%capture
%run ../part_2/part2.ipynb
import math
import numpy as np
import pandas as pd
from collections import defaultdict

In [2]:
%pip install gensim

Note: you may need to restart the kernel to use updated packages.Requirement already satisfied: gensim in c:\Users\marti\Documents\UNI\4\Recuperació Web\IRWA-2026\irwa-search-engine\irwa_venv\Lib\site-packages (4.4.0)




[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. 

In [3]:
#Does the same as the function in part2 search, but now we put the calculation of the ranking in an other function, since we need all the retrived documents to do the score of our engine.
def get_candidates(query):
    """
    Conjunctive (AND) retrieval over the per-field inverted indexes.

    Returns
    -------
    q_terms   : list of unique pre-processed query terms
    docs      : set of doc_ids that contain ALL the query terms (in any field)
    postings  : dict (field, term) -> {doc_id: term frequency in that field}
    term_docs : dict term -> set of doc_ids containing the term in any field
    """
    q_terms = list(dict.fromkeys(build_terms(query)))   # unique, keeps order
    if not q_terms:
        return [], set(), {}, {}

    postings, term_docs = {}, {t: set() for t in q_terms}
    for t in q_terms:
        for field in index:
            tf = {}
            for doc_id, positions in index[field].get(t, []):
                tf[doc_id] = len(positions)
                term_docs[t].add(doc_id)
            postings[(field, t)] = tf

    docs = set.intersection(*term_docs.values())
    return q_terms, docs, postings, term_docs


def top_k_from_scores(scores, top_k):
    """Sort a {doc_id: score} dict (desc) and cut it to top_k. Ties are broken by doc_id (deterministic)."""
    ranked = sorted(scores.items(), key=lambda x: (-x[1], x[0]))
    return ranked[:top_k]

1 - TF-IDF + cosine similarity

In [4]:
def tf_idf():
    #Calculate document frequency
    df = defaultdict(set) #We use set to only count documents only once
    for field in index:
        for term, postings in index[field].items():
            for doc, _ in postings:
                df[term].add(doc)

    #Calculate IDF, how rare each word is  
    idf = {t: math.log(N / len(d)) for t, d in df.items()}

    #calculate the importance of every term in every document
    doc_vec = [defaultdict(float) for i in range(N)]
    for field, idx in index.items():
        for term, postings in idx.items():
            for doc, pos in postings:
                doc_vec[doc][term] += weights[field] * (1 + math.log(len(pos))) * idf[term]

    #L2 norm (length) of every document vector, as we will need it for the cosine similarity
    norms = np.array([np.linalg.norm(list(d.values())) for d in doc_vec])
    
    return idf, doc_vec, norms


idf_tfidf, doc_vec_tfidf, doc_norm_tfidf = tf_idf()


def cosine_similarity(query, top_k=20):

    terms, docs, i, j= get_candidates(query)
    if not docs:
        return []

    q_tf = {t: terms.count(t) for t in terms}
    q = {t: (1 + math.log(f)) * idf_tfidf.get(t, 0) for t, f in q_tf.items()}
    q_norm = np.linalg.norm(list(q.values()))

    if q_norm == 0:
        return top_k_from_scores({d: 0 for d in docs}, top_k)

    #Calculate cosine similarity
    scores = {
        d: sum(q[t] * doc_vec_tfidf[d].get(t, 0) for t in q) /
           (q_norm * doc_norm_tfidf[d]) if doc_norm_tfidf[d] else 0
        for d in docs
    }

    return top_k_from_scores(scores, top_k)


b. BM25 - how relevant each document is to a query

In [5]:
#Calculate the length of each field
field_len = {f: np.array([len(t) for t in field_terms[f]], dtype=float) for f in field_terms}

#Calculate average field length
avg_len = {f: max(field_len[f].mean(), 1e-9) for f in field_len}

#K1 = Controls how quickly the benefit of repeated terms saturates. More repetitions eventually add less value.
#B = Controls how strongly document length affects the score
K1, B = 1.2, 0.75


def bm25_scores(query, k1=K1, b=B):
    """Returns (docs, {doc_id: BM25 score}) for the conjunctive candidates of the query."""
    q_terms, docs, postings, term_docs = get_candidates(query)
    scores = {}
    for d in docs:
        s = 0.0
        for t in q_terms:
            #weighted term frequency
            tf_tilde = 0.0
            for f in index:
                tf = postings[(f, t)].get(d, 0)
                if tf and weights[f] > 0:
                    tf_tilde += weights[f] * tf / (1 - b + b * field_len[f][d] / avg_len[f])

            #Calculate idf
            n = len(term_docs[t])
            idf = math.log((N - n + 0.5) / (n + 0.5) + 1)

            #Calculate the BM25 contribution
            s += idf * tf_tilde * (k1 + 1) / (tf_tilde + k1)
            
        scores[d] = s
    return scores


def rank_bm25(query, top_k=20, k1=K1, b=B):
    """Conjunctive search ranked by (field-weighted) BM25."""
    return top_k_from_scores(bm25_scores(query, k1, b), top_k)

In [6]:

def to_number(series):
    return pd.to_numeric(
        series.astype(str).str.replace(r"[^0-9.]", "", regex=True),
        errors="coerce"
    )


# Normalize product features
rating = (to_number(df["average_rating"]).fillna(
    to_number(df["average_rating"]).median()) / 5).clip(0, 1).to_numpy()

discount = (to_number(df["discount"]).fillna(
    to_number(df["discount"]).median()) / 100).clip(0, 1).to_numpy()

in_stock = (~df["out_of_stock"].astype(str).str.lower()
            .isin(["true", "1"])).to_numpy()

# Ranking parameters
ALPHA, BETA, GAMMA = 0.2, 0.1, 0.6


def rank_custom(query, top_k=20):
    bm25 = bm25_scores(query)
    if not bm25:
        return []

    max_score = max(bm25.values()) or 1.0

    scores = {}
    for d, s in bm25.items():
        boost = 1 + ALPHA * rating[d] + BETA * discount[d]
        penalty = GAMMA if not in_stock[d] else 1.0
        scores[d] = (s / max_score) * boost * penalty

    return top_k_from_scores(scores, top_k)


c. Our Score

In [7]:
def show_ranking(rank_fn, query, top_k=5):
    for rank, (d, s) in enumerate(rank_fn(query, top_k), start=1):
        print(f"{rank:>2}. {s:7.3f} | {pids[d]} | {df['title'].iat[d]} | {df['brand'].iat[d]} | rating={df['average_rating'].iat[d]}")

for name, fn in [("TF-IDF cosine", cosine_similarity), ("BM25", rank_bm25), ("Custom", rank_custom)]:
    print(f"\n=== {name} ===")
    show_ranking(fn, test_queries[1])


=== TF-IDF cosine ===
 1.   0.613 | TSHFUSKPWUZSP6F6 | Printed Men Round Neck Black T-Shirt | She | rating=3
 2.   0.545 | TSHFZWNF4YHDDGYE | Printed Men Round Neck Black T-Shirt | Styleska | rating=3.9
 3.   0.539 | TSHFV5YGHGYGVQEF | Printed Men Round Neck Black T-Shirt | Pol | rating=3.8
 4.   0.537 | TSHFXYNGKBM82EZR | Printed Men Round Neck Black T-Shirt | CA | rating=1.7
 5.   0.534 | TSHFXU28NUVZNDQA | Printed Men Round Neck Black T-Shirt | Styleska | rating=3.9

=== BM25 ===
 1.   6.339 | TSHFWF58KHZT8BBX | Solid Men Round Neck Blue T-Shirt | MASH UNLIMIT | rating=3.2
 2.   6.330 | TSHFWF57CHKBUNHE | Solid Men Round Neck White T-Shirt | MASH UNLIMIT | rating=3.2
 3.   6.322 | TSHFWF589Z4GGVJ2 | Solid Men Round Neck Red T-Shirt | MASH UNLIMIT | rating=3.2
 4.   6.318 | TSHFVXGQG7SSCQFH | Printed Men Round Neck Black T-Shirt | MASH UNLIMIT | rating=3.2
 5.   6.318 | TSHFVXGQ8Z5ZHCBS | Printed Men Round Neck Yellow T-Shirt | MASH UNLIMIT | rating=3.2

=== Custom ===
 1.   1.216 |

In [8]:
def evaluate_ranker(rank_fn, k=10):
    all_y, rrs, rows = [], [], []

    for qid, q in validation_queries.items():
        ql = labels_df[labels_df["query_id"] == qid]
        rel = dict(zip(ql["pid"], ql["labels"]))

        ranked = rank_fn(q, top_k=len(pids))

        retrieved = [
            pids[d] for d, _ in ranked if pids[d] in rel
        ][:k]

        y = [int(rel[p]) for p in retrieved]
        total = int((ql["labels"] == 1).sum())

        rr = rr_at_k(y, k)

        all_y.append(y)
        rrs.append(rr)

        rows.append({
            "P@10": precision_at_k(y, k),
            "R@10": recall_at_k(y, total, k),
            "F1@10": f1_at_k(y, total, k),
            "AP@10": avg_precision_at_k(y, k),
            "RR@10": rr,
            "NDCG@10": ndcg_at_k(y, k, n_relevant=total)
        })

    res = pd.DataFrame(rows)
    res.loc["mean"] = res.mean()

    res.loc["mean", "MAP@10"] = map_at_k(all_y, k)[0]
    res.loc["mean", "MRR@10"] = mrr_at_k(rrs)

    return res

for name, fn in [
    ("TF-IDF cosine", cosine_similarity),
    ("BM25", rank_bm25),
    ("Custom", rank_custom)
]:
    print(f"\n=== {name} ===")
    display(evaluate_ranker(fn).round(3))


=== TF-IDF cosine ===


,P@10,R@10,F1@10,AP@10,RR@10,NDCG@10,MAP@10,MRR@10
0,1.0,0.769,0.870,1.000,1.0,1.000,NaN,NaN
1,0.6,0.600,0.600,0.948,1.0,0.716,NaN,NaN
mean,0.8,0.685,0.735,0.974,1.0,0.858,0.974,1.0



=== BM25 ===


,P@10,R@10,F1@10,AP@10,RR@10,NDCG@10,MAP@10,MRR@10
0,1.0,0.769,0.870,1.000,1.0,1.000,NaN,NaN
1,0.6,0.600,0.600,0.931,1.0,0.712,NaN,NaN
mean,0.8,0.685,0.735,0.965,1.0,0.856,0.965,1.0



=== Custom ===


,P@10,R@10,F1@10,AP@10,RR@10,NDCG@10,MAP@10,MRR@10
0,0.90,0.692,0.783,0.977,1.0,0.931,NaN,NaN
1,0.60,0.600,0.600,0.948,1.0,0.716,NaN,NaN
mean,0.75,0.646,0.691,0.962,1.0,0.823,0.962,1.0


## 2.

In [9]:
from gensim.models import Word2Vec

def doc_tokens(i):
    """All the tokens of document i that are used to represent it (seller excluded)."""
    return (field_terms["title"][i] + field_terms["description"][i] + field_terms["category"][i]
            + field_terms["brand"][i] + field_terms["details"][i])

corpus = [doc_tokens(i) for i in range(N)]

w2v = Word2Vec(sentences=corpus, vector_size=100, window=5, min_count=2, sg=1,
               epochs=20, workers=1, seed=42)      # workers=1 + seed -> reproducible

def text_vector(tokens):
    """Average of the word2vec vectors of the known tokens (zero vector if none is known)."""
    vecs = [w2v.wv[t] for t in tokens if t in w2v.wv] #obtain the vector of each token in the text, if it is in the vocabulary of the model
    return np.mean(vecs, axis=0) if vecs else np.zeros(w2v.vector_size) #return the average of the vectors, if there are no known tokens return a zero vector

doc_vecs = np.vstack([text_vector(t) for t in corpus]) #create a matrix where each row is the vector representation of a document, by applying the text_vector function to each document in the corpus
doc_vec_norms = np.linalg.norm(doc_vecs, axis=1) #compute the norm of each document vector

In [10]:
def rank_word2vec(query, top_k=20):
    """Conjunctive search; candidates ranked by cosine(avg w2v(query), avg w2v(document))."""
    q_terms, docs, _, _ = get_candidates(query) #we only need the query terms and the documents that contain all the query terms
    if not docs:
        return []

    #Compute the vector representation of the query and its norm
    q_vec = text_vector(q_terms)
    q_norm = np.linalg.norm(q_vec)
    scores = {}

    for d in docs: #for every document that contains all the query terms
        #we compute the cosine similarity between the query vector and the document vector
        denom = q_norm * doc_vec_norms[d]
        scores[d] = float(doc_vecs[d] @ q_vec / denom) if denom > 0 else 0.0
    return top_k_from_scores(scores, top_k) #return the top_k documents with the highest cosine similarity scores

#execute the 5 test queries
w2v_rows = [] #to store the results of the ranking for each query

#for every query in the test_queries dictionary, we rank the documents using the rank_word2vec function and store the results in the w2v_rows list
for qid, q in test_queries.items():
    for rank, (d, s) in enumerate(rank_word2vec(q, top_k=20), start=1):
        w2v_rows.append({"query_id": qid, "query": q, "rank": rank, "pid": pids[d],
                         "title": df["title"].iat[d], "w2v_cosine": round(s, 4)})
w2v_results = pd.DataFrame(w2v_rows)
display(w2v_results)

,query_id,query,rank,pid,title,w2v_cosine
0,1,men round neck cotton shirt,1,TSHFUSKPWUZSP6F6,Printed Men Round Neck Black T-Shirt,0.9168
1,1,men round neck cotton shirt,2,TSHFZT9F4AU9GMHX,Printed Men Round Neck Black T-Shirt,0.9084
2,1,men round neck cotton shirt,3,TSHFVXGQG7SSCQFH,Printed Men Round Neck Black T-Shirt,0.9059
3,1,men round neck cotton shirt,4,TSHFZUDXJNVHW24R,Printed Men Round Neck White T-Shirt,0.9046
4,1,men round neck cotton shirt,5,TSHFW9CRBTZTKCGN,Printed Men Round Neck White T-Shirt,0.9044
...,...,...,...,...,...,...
95,5,men solid polo shirt,16,TSHFYMZQT8M57ZGU,Solid Men Polo Neck White T-Shirt,0.8397
96,5,men solid polo shirt,17,TSHFYMXQHPQZSP5M,Solid Men Polo Neck Grey T-Shirt,0.8386
97,5,men solid polo shirt,18,TSHFNJYBHU8NWCGH,Solid Men Polo Neck Orange T-Shirt,0.8382
98,5,men solid polo shirt,19,TSHFUGZ9ZG5XN2FY,Solid Men Polo Neck Blue T-Shirt,0.8364


In [11]:
# Same metrics on the validation queries, to compare with the other rankers
display(evaluate_ranker(rank_word2vec).round(3))

,P@10,R@10,F1@10,AP@10,RR@10,NDCG@10,MAP@10,MRR@10
0,0.90,0.692,0.783,0.963,1.0,0.927,NaN,NaN
1,0.60,0.600,0.600,0.856,1.0,0.687,NaN,NaN
mean,0.75,0.646,0.691,0.909,1.0,0.807,0.909,1.0


We can observe that it has good results. In first query works better than in the secon since all the metrics are higher. In average, 7.5 out of 10 relevant documents are retrieved and it retrieves a 64.6% of all relevant documents. 